# 03 — Mamba SSM v3: Chunked Scan + Full Timing Instrumentation

**Task ID:** `MODEL-MAMBA-ZERO-FEE-007`  
**Version:** 0.3 — v2 + chunked vectorized SSM scan + tqdm progress bars  
**Date:** 2026-06-01

## What changed vs v2
| # | v2 | v3 |
|---|----|----|
| 1 | Sequential Python SSM loop (L=24 iters/batch) | Chunked vectorized scan (L/8 = 3 carries/batch) ~4-8× faster |
| 2 | Prints every 5 epochs, no ETA | tqdm per-epoch + per-run with live ETA |
| 3 | M1 (1 feature) + M3 (12 fallback V4 features) included | Dropped — only M2 and M4 from actual selection |

## Pipeline
```
V1 (195 features)  +  V4 (25 features)
         │
         ▼
  4-Stage Feature Selection  (cached)
         │
         ▼
  Mamba SSM (d_model=64, 2 layers, d_state=16, chunk=8)
  Trained on M2 + M4 subsets × 2 labels = 4 runs
  + Permutation Importance
```

In [1]:
import json
import math
import time
import warnings
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import QuantileTransformer
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='darkgrid', palette='muted', font_scale=1.0)

if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    print('Device: Apple MPS')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f'Device: CUDA ({torch.cuda.get_device_name()})')
else:
    DEVICE = torch.device('cpu')
    print('Device: CPU')

Device: Apple MPS


/Users/wojciech.neuman/Documents/hybrid-multi-agent-trading-system/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys

def _find_repo_root() -> Path:
    p = Path.cwd()
    while p != p.parent:
        if (p / 'pyproject.toml').exists():
            return p
        p = p.parent
    raise RuntimeError('pyproject.toml not found')

REPO     = _find_repo_root()
RAW_DIR  = REPO / 'data' / 'raw'
FEAT_DIR = REPO / 'data' / 'features'
ARTS_DIR = REPO / 'artifacts' / '03_mamba_omni_0fee_v3'
ARTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO / 'src'))

print(f'Repo: {REPO}')

# ── Sequence constants ────────────────────────────────────────────────────────
SYMBOL      = 'BTCUSDT'
INTERVAL    = '1h'
SEQ_LEN     = 24      # 24h context at 1h bars
STRIDE      = 1       # 1-bar step
BATCH       = 256
EPOCHS      = 50
LR          = 3e-4
FOCAL_GAMMA = 2.0

# ── Chunked SSM scan — key speedup vs v2 ─────────────────────────────────────
# v2: L=24 Python iterations per forward pass (sequential loop)
# v3: L/SCAN_CHUNK = 24/8 = 3 Python carries, rest vectorized via cumsum
SCAN_CHUNK = 8

# ── Feature selection (same as v2, uses cached result from v2 if present) ────
STAGE2_TOP_K        = 60
STAGE3_N_SUBWINDOWS = 3
STAGE3_MIN_WINS     = 2
STAGE4_TOP_K        = 20

# ── Splits (same as v2) ───────────────────────────────────────────────────────
OOS_START = pd.Timestamp('2024-01-01')
TRAIN_END = pd.Timestamp('2022-12-31')
VAL_END   = pd.Timestamp('2023-12-31')

# Cache: try v2 cache first (avoid rerunning feature selection)
V2_ARTS   = REPO / 'artifacts' / '03_mamba_omni_0fee_v2'
SELECTION_CACHE = ARTS_DIR / 'selected_features.json'
V2_SELECTION    = V2_ARTS  / 'selected_features.json'

print(f'SEQ_LEN={SEQ_LEN}  STRIDE={STRIDE}  BATCH={BATCH}  EPOCHS={EPOCHS}  SCAN_CHUNK={SCAN_CHUNK}')
print(f'OOS from {OOS_START.date()}  |  Train≤{TRAIN_END.date()}  Val≤{VAL_END.date()}')

Repo: /Users/wojciech.neuman/Documents/hybrid-multi-agent-trading-system
SEQ_LEN=24  STRIDE=1  BATCH=256  EPOCHS=50  SCAN_CHUNK=8
OOS from 2024-01-01  |  Train≤2022-12-31  Val≤2023-12-31


---
## Phase A — Load V1 + V4 Features

In [3]:
v4_df = pd.read_parquet(FEAT_DIR / 'BTCUSDT_1h_v4_features.parquet')
v4_df.index = v4_df.index.tz_localize(None) if v4_df.index.tz else v4_df.index
V4_FEATURE_COLS = list(v4_df.columns)
print(f'V4: {len(V4_FEATURE_COLS)} features  ({v4_df.index.min().date()} → {v4_df.index.max().date()})')

v1_df = pd.read_parquet(FEAT_DIR / 'BTCUSDT_1h_features.parquet')
v1_df.index = v1_df.index.tz_localize(None) if v1_df.index.tz else v1_df.index

with open(FEAT_DIR / 'feature_registry.json') as f:
    registry = json.load(f)
BACKTEST_COLS = registry.get('backtest_only_cols', [])

_EXCLUDE = set(BACKTEST_COLS) | {
    'open', 'high', 'low', 'close', 'volume',
    'label', 'tbm_label', 'fh_label', 'forward_ret',
}
V1_FEATURE_COLS = [
    c for c in v1_df.columns
    if c not in _EXCLUDE and pd.api.types.is_numeric_dtype(v1_df[c])
]
print(f'V1: {len(V1_FEATURE_COLS)} features  ({v1_df.index.min().date()} → {v1_df.index.max().date()})')

raw_df = pd.read_parquet(RAW_DIR / 'BTCUSDT_1h.parquet',
                          columns=['open', 'high', 'low', 'close', 'volume'])
raw_df.index = raw_df.index.tz_localize(None) if raw_df.index.tz else raw_df.index

merged = v1_df.copy()
for col in V4_FEATURE_COLS:
    merged[col] = v4_df[col].reindex(merged.index)
merged['high']  = raw_df['high'].reindex(merged.index)
merged['low']   = raw_df['low'].reindex(merged.index)
merged['close'] = raw_df['close'].reindex(merged.index)
merged.index    = merged.index.tz_localize(None) if merged.index.tz else merged.index

ALL_FEATURE_COLS = V1_FEATURE_COLS + V4_FEATURE_COLS
print(f'\nMerged: {len(merged):,} rows  |  {len(ALL_FEATURE_COLS)} features (V1+V4)')
print(f'  Range: {merged.index.min().date()} → {merged.index.max().date()}')

V4: 25 features  (2017-11-15 → 2026-05-16)
V1: 195 features  (2017-11-15 → 2026-05-16)

Merged: 74,366 rows  |  220 features (V1+V4)
  Range: 2017-11-15 → 2026-05-16


---
## Phase B — 4-Stage Feature Selection

Same pipeline as v2. **Tries v2 cache first** to avoid rerunning selection.

In [4]:
train_sel_mask = merged.index < OOS_START
df_sel = merged[train_sel_mask][ALL_FEATURE_COLS].copy()
y_sel  = (raw_df['close'].reindex(merged.index).pct_change(1).shift(-1)[train_sel_mask] > 0).astype(int).values
X_sel  = df_sel.fillna(0).values.astype(np.float32)
print(f'Selection set: {X_sel.shape[0]:,} bars × {X_sel.shape[1]} features')


def _stage1(X, cols, var_thresh=1e-6, rho_thresh=0.85):
    from scipy.stats import spearmanr
    keep = [c for c, v in zip(cols, X.var(axis=0)) if v > var_thresh]
    idx  = [cols.index(c) for c in keep]
    X1   = X[:, idx]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        corr, _ = spearmanr(X1)
    if X1.shape[1] == 1:
        return keep, X1
    corr = np.abs(np.array(corr))
    np.fill_diagonal(corr, 0.0)
    removed = set()
    for i in range(len(keep)):
        if i in removed: continue
        for j in range(i + 1, len(keep)):
            if j not in removed and corr[i, j] > rho_thresh:
                removed.add(j)
    survive = [c for i, c in enumerate(keep) if i not in removed]
    idx2    = [keep.index(c) for c in survive]
    return survive, X1[:, idx2]

def _stage2(X, y, cols, top_k=STAGE2_TOP_K):
    mi     = mutual_info_classif(X, y, discrete_features=False, random_state=42)
    ranked = sorted(zip(cols, mi), key=lambda t: t[1], reverse=True)
    top    = [c for c, _ in ranked[:top_k]]
    idx    = [cols.index(c) for c in top]
    return top, X[:, idx]

def _stage3(X, y, cols, n_win=STAGE3_N_SUBWINDOWS, min_wins=STAGE3_MIN_WINS,
            top_k=STAGE2_TOP_K):
    n      = len(X)
    sub_sz = n // n_win
    counts = {c: 0 for c in cols}
    for w in range(n_win):
        sl = slice(w * sub_sz, (w + 1) * sub_sz if w < n_win - 1 else n)
        Xw, yw = X[sl], y[sl]
        if len(np.unique(yw)) < 2: continue
        mi  = mutual_info_classif(Xw, yw, discrete_features=False, random_state=42)
        top = sorted(zip(cols, mi), key=lambda t: t[1], reverse=True)[:top_k]
        for c, _ in top: counts[c] += 1
    survive = [c for c in cols if counts[c] >= min_wins]
    if len(survive) < STAGE4_TOP_K:
        survive = cols
    idx = [cols.index(c) for c in survive]
    return survive, X[:, idx]

def _logloss(probs, y):
    probs = np.clip(probs, 1e-7, 1.0 - 1e-7)
    return float(-np.mean(y * np.log(probs[:, 1]) + (1 - y) * np.log(probs[:, 0])))

def _stage4(X, y, cols, top_k=STAGE4_TOP_K, n_repeats=3):
    n_val = max(int(0.10 * len(X)), 50)
    X_tr, y_tr = X[:-n_val], y[:-n_val]
    X_vl, y_vl = X[-n_val:], y[-n_val:]
    params = dict(objective='binary', metric='binary_logloss', n_estimators=200,
                  num_leaves=31, learning_rate=0.05, verbose=-1, n_jobs=-1,
                  random_state=42)
    model = lgb.LGBMClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)],
              callbacks=[lgb.early_stopping(20, verbose=False)])
    baseline = _logloss(model.predict_proba(X_vl), y_vl)
    imp = np.zeros(len(cols))
    rng = np.random.default_rng(42)
    for _ in range(n_repeats):
        for j in range(len(cols)):
            Xp = X_vl.copy(); rng.shuffle(Xp[:, j])
            imp[j] += _logloss(model.predict_proba(Xp), y_vl) - baseline
    imp /= n_repeats
    ranked = sorted(zip(cols, imp), key=lambda t: t[1], reverse=True)
    return [c for c, _ in ranked[:top_k]]


# Try v2 cache first, then local cache, then rerun
if SELECTION_CACHE.exists():
    src = SELECTION_CACHE
elif V2_SELECTION.exists():
    src = V2_SELECTION
    print(f'Using v2 feature selection cache.')
else:
    src = None

if src is not None:
    with open(src) as f:
        _sel = json.load(f)
    SELECTED_FEATURES = _sel['features']
    # Copy to v3 arts dir
    with open(SELECTION_CACHE, 'w') as f:
        json.dump(_sel, f, indent=2)
    print(f'Loaded {len(SELECTED_FEATURES)} selected features from cache ({src.name}).')
else:
    print('Running 4-stage feature selection on train split...')
    t0_sel = time.time()
    print(f'  Stage 1: variance + Spearman ({len(ALL_FEATURE_COLS)} → ?) ...')
    cols1, X1 = _stage1(X_sel, ALL_FEATURE_COLS)
    print(f'    → {len(cols1)} survivors')
    print(f'  Stage 2: MI ranking ({len(cols1)} → top {STAGE2_TOP_K}) ...')
    cols2, X2 = _stage2(X1, y_sel, cols1)
    print(f'    → {len(cols2)} survivors')
    print(f'  Stage 3: stability filter ({len(cols2)} → ?) ...')
    cols3, X3 = _stage3(X2, y_sel, cols2)
    print(f'    → {len(cols3)} survivors')
    print(f'  Stage 4: LGBM permutation pruning ({len(cols3)} → top {STAGE4_TOP_K}) ...')
    SELECTED_FEATURES = _stage4(X3, y_sel, cols3)
    print(f'    → {len(SELECTED_FEATURES)} final features')
    print(f'\nSelection done in {time.time()-t0_sel:.0f}s')
    with open(SELECTION_CACHE, 'w') as f:
        json.dump({'features': SELECTED_FEATURES,
                   'stage_counts': [len(ALL_FEATURE_COLS), len(cols1),
                                    len(cols2), len(cols3), len(SELECTED_FEATURES)]}, f, indent=2)

print(f'\nSelected features ({len(SELECTED_FEATURES)}):')
for i, feat in enumerate(SELECTED_FEATURES, 1):
    is_v4 = ' [V4]' if feat in V4_FEATURE_COLS else ''
    print(f'  {i:2d}. {feat}{is_v4}')

Selection set: 53,581 bars × 220 features
Using v2 feature selection cache.
Loaded 20 selected features from cache (selected_features.json).

Selected features (20):
   1. close_vs_true_vwap [V4]
   2. ret_2h
   3. close_vs_ema_7
   4. stoch_k_14
   5. rsi_vol_confirm
   6. hour_sin
   7. range_vs_atr
   8. candles_since_cross
   9. ret_3h
  10. close_vs_s1
  11. bull_streak
  12. vw_rsi_14
  13. candle_body
  14. macd_hist_5_13
  15. hl_position_24h
  16. ad_z_168h
  17. hurst_168h
  18. bb_squeeze_50
  19. ret_24h
  20. macd_divergence


In [5]:
# v3: only subsets derived from actual feature selection (M2 + M4)
# M1 (1 feature) and M3 (12 fallback V4 regime features) dropped — not from selection

def _intersect(cols):
    return [c for c in cols if c in merged.columns]

M2_FEATURES = _intersect([c for c in SELECTED_FEATURES if c in V1_FEATURE_COLS]
                          or [c for c in SELECTED_FEATURES if c not in V4_FEATURE_COLS])
M4_FEATURES = _intersect(SELECTED_FEATURES)

FEAT_SUBSETS = {
    'M2_v1_selected': M2_FEATURES,
    'M4_omni':        M4_FEATURES,
}

for name, cols in FEAT_SUBSETS.items():
    v4_count = sum(c in V4_FEATURE_COLS for c in cols)
    print(f'{name}: {len(cols)} features  ({v4_count} V4)')

M2_v1_selected: 19 features  (0 V4)
M4_omni: 20 features  (1 V4)


---
## Phase C — Label Generation

In [6]:
close_s = raw_df['close'].reindex(merged.index)
close_s.index = close_s.index.tz_localize(None) if close_s.index.tz else close_s.index

y_fh = (close_s.pct_change(1).shift(-1) > 0).astype(int)
y_fh.name = 'label_fh'

hi_s = raw_df['high'].reindex(merged.index)
lo_s = raw_df['low'].reindex(merged.index)
hi_s.index = lo_s.index = close_s.index
tr     = pd.concat([
    hi_s - lo_s,
    (hi_s - close_s.shift(1)).abs(),
    (lo_s - close_s.shift(1)).abs(),
], axis=1).max(axis=1)
atr_1h = tr.ewm(span=14, adjust=False).mean()

SL_MULT  = 1.5
TP_MULT  = 1.5
MAX_HOLD = SEQ_LEN

close_arr = close_s.values
atr_arr   = atr_1h.values
n         = len(close_arr)
y_tbm_arr = np.full(n, np.nan)

print(f'Computing TBM labels (n={n:,}, max_hold={MAX_HOLD}) ...')
t0 = time.time()
for i in range(n - MAX_HOLD):
    if np.isnan(atr_arr[i]) or atr_arr[i] == 0:
        continue
    entry = close_arr[i]
    tp    = entry + TP_MULT * atr_arr[i]
    sl    = entry - SL_MULT * atr_arr[i]
    for j in range(i + 1, min(i + MAX_HOLD + 1, n)):
        if close_arr[j] >= tp:   y_tbm_arr[i] = 1; break
        if close_arr[j] <= sl:   y_tbm_arr[i] = 0; break

y_tbm = pd.Series(y_tbm_arr, index=close_s.index, name='label_tbm')
print(f'TBM done in {time.time()-t0:.0f}s  |  positive rate: {np.nanmean(y_tbm_arr):.3f}')
print(f'FH  positive rate: {y_fh.mean():.3f}')

Computing TBM labels (n=74,366, max_hold=24) ...
TBM done in 0s  |  positive rate: 0.511
FH  positive rate: 0.509


In [7]:
df_base = merged[ALL_FEATURE_COLS].copy()
df_base['label_fh']  = y_fh.reindex(df_base.index)
df_base['label_tbm'] = y_tbm.reindex(df_base.index)
df_base['close']     = close_s.reindex(df_base.index)
df_base = df_base.dropna(subset=['close'])
df_base.index = df_base.index.tz_localize(None) if df_base.index.tz else df_base.index

train_mask = df_base.index <= TRAIN_END
val_mask   = (df_base.index > TRAIN_END) & (df_base.index <= VAL_END)
oos_mask   = df_base.index > VAL_END

df_tr  = df_base[train_mask].copy()
df_vl  = df_base[val_mask].copy()
df_oos = df_base[oos_mask].copy()

print(f'Train: {len(df_tr):,}  ({df_tr.index.min().date()} → {df_tr.index.max().date()})')
print(f'Val:   {len(df_vl):,}  ({df_vl.index.min().date()} → {df_vl.index.max().date()})')
print(f'OOS:   {len(df_oos):,}  ({df_oos.index.min().date()} → {df_oos.index.max().date()})')

Train: 44,799  (2017-11-15 → 2022-12-31)
Val:   8,759  (2022-12-31 → 2023-12-31)
OOS:   20,808  (2023-12-31 → 2026-05-16)


---
## Phase D — Normalization & Sequence Dataset

In [8]:
class SequenceDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray, seq_len: int, stride: int = 1):
        self.X       = X.astype(np.float32)
        self.y       = y
        self.seq_len = seq_len
        ends         = np.arange(seq_len - 1, len(X), stride)
        nan_frac     = np.array([np.isnan(X[e-seq_len+1:e+1]).mean() for e in ends])
        valid        = ~np.isnan(y[ends].astype(float)) & (nan_frac <= 0.10)
        self.ends    = ends[valid]

    def __len__(self): return len(self.ends)

    def __getitem__(self, idx):
        e = self.ends[idx]; s = e - self.seq_len + 1
        x = torch.from_numpy(np.nan_to_num(self.X[s:e+1], nan=0.0))
        return x, torch.tensor(int(self.y[e]), dtype=torch.long)


def make_loaders(feat_cols, target_col, batch=BATCH, seq_len=SEQ_LEN, stride=STRIDE):
    qt   = QuantileTransformer(n_quantiles=1000, output_distribution='normal', random_state=42)
    X_tr = qt.fit_transform(df_tr[feat_cols].fillna(0).values)
    X_vl = qt.transform(df_vl[feat_cols].fillna(0).values)
    y_tr = df_tr[target_col].values
    y_vl = df_vl[target_col].values
    ds_tr = SequenceDataset(X_tr, y_tr, seq_len, stride)
    ds_vl = SequenceDataset(X_vl, y_vl, seq_len, stride=1)
    return (
        DataLoader(ds_tr, batch_size=batch, shuffle=True,  drop_last=True),
        DataLoader(ds_vl, batch_size=batch, shuffle=False, drop_last=False),
        qt, len(feat_cols),
    )

print('SequenceDataset and make_loaders ready.')

SequenceDataset and make_loaders ready.


---
## Phase E — Mamba Architecture (Chunked Vectorized Scan)

Key change vs v2: `_chunked_ssm_scan` replaces the sequential `for t in range(L)` Python loop.
- v2: 24 Python iterations per forward pass (one per timestep)  
- v3: L/chunk = 24/8 = **3 Python carries**, remainder vectorized via `torch.cumsum`

Within each chunk the recurrence is solved in closed form:  
`h_t = P_t ⊙ h_carry + P_t ⊙ cumsum(dBu/P, dim=time)`,  where `P_t = ∏ exp(δ⊗A)`

In [9]:
def _chunked_ssm_scan(
    x:      torch.Tensor,   # (B, L, D_inner)
    delta:  torch.Tensor,   # (B, L, D_inner)  — positive timescales
    A:      torch.Tensor,   # (D_inner, N)      — negative definite
    B_mat:  torch.Tensor,   # (B, L, N)
    C_mat:  torch.Tensor,   # (B, L, N)
    D_coef: torch.Tensor,   # (D_inner,)
    chunk:  int = SCAN_CHUNK,
) -> torch.Tensor:
    B, L, D = x.shape
    N = A.shape[-1]

    log_dA = delta.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0)  # (B, L, D, N)
    dBu    = delta.unsqueeze(-1) * B_mat.unsqueeze(2) * x.unsqueeze(-1)  # (B, L, D, N)

    n_chunks = (L + chunk - 1) // chunk
    L_pad    = n_chunks * chunk
    if L_pad > L:
        pad = L_pad - L
        log_dA = F.pad(log_dA, (0, 0, 0, 0, 0, pad))
        dBu    = F.pad(dBu,    (0, 0, 0, 0, 0, pad))
        C_pad  = F.pad(C_mat,  (0, 0, 0, pad))
    else:
        C_pad = C_mat

    log_dA_c = log_dA.view(B, n_chunks, chunk, D, N)
    dBu_c    = dBu.view(B, n_chunks, chunk, D, N)
    C_c      = C_pad.view(B, n_chunks, chunk, N)

    log_P_c      = torch.cumsum(log_dA_c, dim=2)
    P_c          = torch.exp(log_P_c)
    dBu_over_P_c = dBu_c / P_c.clamp(min=1e-12)
    within_c     = P_c * torch.cumsum(dBu_over_P_c, dim=2)
    P_full_c     = torch.exp(log_dA_c.sum(dim=2))

    y = x.new_zeros(B, L_pad, D)
    h = x.new_zeros(B, D, N)
    for ci in range(n_chunks):
        h_in_chunk = P_c[:, ci] * h.unsqueeze(1) + within_c[:, ci]
        y[:, ci*chunk:(ci+1)*chunk] = (
            h_in_chunk * C_c[:, ci].unsqueeze(-2)
        ).sum(-1)
        h = P_full_c[:, ci] * h + within_c[:, ci, -1]

    return y[:, :L] + x * D_coef.unsqueeze(0).unsqueeze(0)


class MambaBlock(nn.Module):
    def __init__(self, d_model: int, d_state: int = 16, d_conv: int = 4,
                 expand: int = 2, chunk: int = SCAN_CHUNK):
        super().__init__()
        d_inner     = d_model * expand
        dt_rank     = max(1, d_model // 16)
        self.d_inner = d_inner
        self.d_state = d_state
        self.dt_rank = dt_rank
        self.chunk   = chunk
        A_init       = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(d_inner, 1)
        self.A_log   = nn.Parameter(torch.log(A_init))
        self.D       = nn.Parameter(torch.ones(d_inner))
        self.norm    = nn.LayerNorm(d_model)
        self.in_proj = nn.Linear(d_model, d_inner * 2, bias=False)
        self.conv1d  = nn.Conv1d(d_inner, d_inner, d_conv, padding=d_conv - 1,
                                  groups=d_inner, bias=True)
        self.x_proj  = nn.Linear(d_inner, dt_rank + d_state * 2, bias=False)
        self.dt_proj = nn.Linear(dt_rank, d_inner, bias=True)
        with torch.no_grad():
            nn.init.constant_(self.dt_proj.bias, math.log(math.expm1(1.0)))
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x        = self.norm(x)
        L        = x.shape[1]
        xz       = self.in_proj(x)
        x_in, z  = xz.chunk(2, dim=-1)
        x_in     = self.conv1d(x_in.transpose(1, 2))[..., :L].transpose(1, 2)
        x_in     = F.silu(x_in)
        dBC      = self.x_proj(x_in)
        dt, B_ssm, C = dBC.split([self.dt_rank, self.d_state, self.d_state], dim=-1)
        delta    = F.softplus(self.dt_proj(dt))
        A        = -torch.exp(self.A_log.float())
        y        = _chunked_ssm_scan(x_in, delta, A, B_ssm, C, self.D, self.chunk)
        return self.out_proj(y * F.silu(z)) + residual


class MambaClassifier(nn.Module):
    def __init__(self, n_features: int, d_model: int = 64, n_layers: int = 2,
                 d_state: int = 16, d_conv: int = 4, expand: int = 2,
                 n_classes: int = 2, dropout: float = 0.1):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.blocks     = nn.ModuleList([
            MambaBlock(d_model, d_state, d_conv, expand) for _ in range(n_layers)
        ])
        self.dropout    = nn.Dropout(dropout)
        self.head       = nn.Linear(d_model, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_proj(x)
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.dropout(x[:, -1, :]))


# ── Warm-up timing benchmark ──────────────────────────────────────────────────
_m = MambaClassifier(n_features=20).to(DEVICE)
_x = torch.randn(BATCH, SEQ_LEN, 20).to(DEVICE)
_m(_x)  # warm-up
t0 = time.perf_counter()
for _ in range(10): _m(_x)
ms_fwd = (time.perf_counter() - t0) / 10 * 1000
ms_batch = ms_fwd * 3  # fwd + bwd estimate
print(f'MambaClassifier: {sum(p.numel() for p in _m.parameters()):,} params')
print(f'Scan: chunked (chunk={SCAN_CHUNK}, {SEQ_LEN // SCAN_CHUNK} carries/pass vs {SEQ_LEN} in v2)')
print(f'Forward  ({BATCH}×{SEQ_LEN}×20 on {DEVICE}): {ms_fwd:.1f} ms/batch')
print(f'Fwd+bwd  estimate: {ms_batch:.1f} ms/batch')
del _m, _x

MambaClassifier: 67,010 params
Scan: chunked (chunk=8, 3 carries/pass vs 24 in v2)
Forward  (256×24×20 on mps): 3.5 ms/batch
Fwd+bwd  estimate: 10.5 ms/batch


---
## Phase F — Training Utilities + Focal Loss

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0, alpha: torch.Tensor = None):
        super().__init__()
        self.gamma = gamma
        self.register_buffer('alpha', alpha)

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none', weight=self.alpha)
        return ((1 - torch.exp(-ce)) ** self.gamma * ce).mean()


def make_focal_loss(y_train, gamma=FOCAL_GAMMA, device=DEVICE):
    counts = np.bincount(y_train[~np.isnan(y_train)].astype(int), minlength=2)
    alpha  = torch.tensor(1.0 / (counts + 1), dtype=torch.float32).to(device)
    return FocalLoss(gamma=gamma, alpha=alpha / alpha.sum())


def train_one_epoch(model, loader, criterion, opt, device):
    model.train()
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, device):
    model.eval()
    all_p, all_y = [], []
    for xb, yb in loader:
        p = torch.softmax(model(xb.to(device)), dim=-1)[:, 1].cpu().numpy()
        all_p.append(p); all_y.append(yb.numpy())
    p = np.concatenate(all_p); y = np.concatenate(all_y)
    return p, y, (roc_auc_score(y, p) if len(np.unique(y)) >= 2 else 0.5)

print('FocalLoss + training utilities ready.')

FocalLoss + training utilities ready.


: 

---
## Phase G — Training (M2 + M4 × 2 Targets = 4 Runs)

Progress bars show:  
- **Outer bar**: overall run progress (X/4 runs)  
- **Inner bar**: epoch progress within each run, with live loss/AUC and ETA  
- After every epoch: elapsed time, per-epoch speed, projected finish time

In [ ]:
RESULTS = []
_session_start = time.time()
_run_times     = []  # seconds per completed run, used for ETA

n_total_runs = sum(1 for feat_cols in FEAT_SUBSETS.values() if feat_cols) * 2

run_bar = tqdm(total=n_total_runs, desc='Phase G — all runs', unit='run',
               bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]')

for subset_name, feat_cols in FEAT_SUBSETS.items():
    if not feat_cols:
        print(f'SKIP {subset_name}: empty feature list')
        continue
    for target_col in ['label_fh', 'label_tbm']:
        run_id      = f'{subset_name}__{target_col}'
        cache_probs = ARTS_DIR / f'probs_{run_id}.npy'
        cache_meta  = ARTS_DIR / f'meta_{run_id}.json'

        if cache_probs.exists() and cache_meta.exists():
            run_bar.write(f'[{run_id}] Loading from cache.')
            val_probs = np.load(cache_probs)
            with open(cache_meta) as f:
                meta = json.load(f)
            RESULTS.append({'run_id': run_id, 'feat_key': subset_name,
                            'target': target_col, 'val_probs': val_probs, **meta})
            _run_times.append(meta.get('elapsed_s', 0))
            run_bar.update(1)
            continue

        run_bar.write(f'\n{"="*65}')
        run_bar.write(f'RUN: {run_id}  ({len(feat_cols)} features)')
        run_bar.write(f'{"="*65}')

        tr_loader, vl_loader, qt, n_feat = make_loaders(feat_cols, target_col)
        if len(tr_loader) == 0:
            run_bar.write('  SKIP: empty training loader')
            run_bar.update(1)
            continue

        model     = MambaClassifier(n_features=n_feat).to(DEVICE)
        n_params  = sum(p.numel() for p in model.parameters())
        run_bar.write(f'  Parameters: {n_params:,}  |  train batches/epoch: {len(tr_loader)}')

        # ETA hint from prior runs
        if _run_times:
            avg_run_s = np.mean(_run_times)
            remaining_runs = n_total_runs - run_bar.n
            run_bar.write(
                f'  Avg run so far: {avg_run_s/60:.1f} min  |  '
                f'Est. remaining: {avg_run_s * remaining_runs / 60:.0f} min  '
                f'({avg_run_s * remaining_runs / 3600:.1f} h)'
            )

        y_tr_vals = df_tr[target_col].dropna().values.astype(int)
        criterion = make_focal_loss(y_tr_vals)
        opt       = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        sched     = SequentialLR(opt,
                        [LinearLR(opt, start_factor=0.1, end_factor=1.0, total_iters=5),
                         CosineAnnealingLR(opt, T_max=EPOCHS - 5, eta_min=1e-6)],
                        milestones=[5])

        best_auc = best_epoch = 0
        patience = 10
        train_aucs, val_aucs = [], []
        t_run = time.time()

        ep_bar = tqdm(range(1, EPOCHS + 1), desc=f'  {run_id}', unit='ep', leave=False,
                      bar_format='  {l_bar}{bar}| ep {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]')

        for epoch in ep_bar:
            tr_loss                 = train_one_epoch(model, tr_loader, criterion, opt, DEVICE)
            _, _, tr_auc            = eval_epoch(model, tr_loader, DEVICE)
            vl_probs, vl_y, vl_auc = eval_epoch(model, vl_loader, DEVICE)
            sched.step()
            train_aucs.append(tr_auc); val_aucs.append(vl_auc)

            elapsed  = time.time() - t_run
            ep_speed = elapsed / epoch           # s/epoch
            ep_bar.set_postfix({
                'loss': f'{tr_loss:.4f}',
                'tr':   f'{tr_auc:.4f}',
                'val':  f'{vl_auc:.4f}',
                'best': f'{best_auc:.4f}',
                's/ep': f'{ep_speed:.1f}',
            })

            if vl_auc > best_auc:
                best_auc = vl_auc; best_epoch = epoch
                best_probs = vl_probs.copy()
                torch.save(model.state_dict(), ARTS_DIR / f'model_{run_id}.pt')

            if epoch - best_epoch >= patience:
                ep_bar.write(f'    Early stop ep {epoch}  (best={best_auc:.4f} @ ep {best_epoch})')
                break

        ep_bar.close()
        elapsed = time.time() - t_run
        _run_times.append(elapsed)

        run_bar.write(
            f'  Done: best val AUC={best_auc:.4f} @ ep {best_epoch}  '
            f'| {elapsed:.0f}s ({elapsed/60:.1f} min)  '
            f'| {elapsed/best_epoch:.1f} s/ep'
        )
        # Session-level timing summary
        session_elapsed = time.time() - _session_start
        runs_done  = run_bar.n + 1
        runs_left  = n_total_runs - runs_done
        eta_s      = np.mean(_run_times) * runs_left
        run_bar.write(
            f'  Session: {session_elapsed/60:.0f} min elapsed  |  '
            f'~{eta_s/60:.0f} min ({eta_s/3600:.1f} h) remaining for {runs_left} run(s)'
        )

        np.save(cache_probs, best_probs)
        meta = {'best_auc': float(best_auc), 'best_epoch': int(best_epoch),
                'train_aucs': train_aucs, 'val_aucs': val_aucs,
                'n_features': int(n_feat), 'n_train_seqs': len(tr_loader.dataset),
                'n_val_seqs': len(vl_loader.dataset), 'elapsed_s': float(elapsed),
                'feat_cols': feat_cols}
        with open(cache_meta, 'w') as f:
            json.dump(meta, f, indent=2)
        RESULTS.append({'run_id': run_id, 'feat_key': subset_name,
                        'target': target_col, 'val_probs': best_probs, **meta})

        run_bar.update(1)

run_bar.close()
print(f'\nTraining complete: {len(RESULTS)} runs  |  Total: {(time.time()-_session_start)/60:.1f} min')

Phase G — all runs:   0%|          | 0/4 [00:00<?]


RUN: M2_v1_selected__label_fh  (19 features)


Phase G — all runs:   0%|          | 0/4 [00:00<?]

  Parameters: 66,946  |  train batches/epoch: 174


In [ ]:
rows = [{'Run': r['run_id'].replace('__', ' | '), 'Features': r['n_features'],
         'Val AUC': round(r['best_auc'], 4), 'Best Ep': r['best_epoch'],
         'Train Seqs': r['n_train_seqs'], 'Val Seqs': r['n_val_seqs'],
         'Time (min)': round(r['elapsed_s'] / 60, 1)}
        for r in RESULTS]
print(pd.DataFrame(rows).sort_values('Val AUC', ascending=False).to_string(index=False))
best_run = max(RESULTS, key=lambda r: r['best_auc']) if RESULTS else None
if best_run:
    print(f'\nBest: {best_run["run_id"]}  AUC={best_run["best_auc"]:.4f}')

---
## Phase H — Feature Attribution (Permutation Importance)

In [ ]:
def compute_sequence_permutation_importance(
    model, dataloader, base_auc, device, feature_names, n_repeats=3
):
    model.eval()
    all_x, all_y = [], []
    with torch.no_grad():
        for xb, yb in dataloader:
            all_x.append(xb); all_y.append(yb)
    X_clean = torch.cat(all_x, dim=0).to(device)
    Y_true  = torch.cat(all_y, dim=0).cpu().numpy()
    importances = {}
    feat_bar = tqdm(feature_names, desc='  Perm. importance', leave=False, unit='feat')
    for fname in feat_bar:
        f_idx = feature_names.index(fname)
        feat_bar.set_postfix({'feature': fname})
        drops = []
        for _ in range(n_repeats):
            Xp = X_clean.clone()
            Xp[:, :, f_idx] = Xp[torch.randperm(Xp.shape[0]), :, f_idx]
            preds = []
            with torch.no_grad():
                for chunk in torch.split(Xp, 256, dim=0):
                    preds.append(torch.softmax(model(chunk), dim=-1)[:, 1].cpu())
            Yp = torch.cat(preds).numpy()
            if len(np.unique(Y_true)) >= 2:
                drops.append(base_auc - roc_auc_score(Y_true, Yp))
        importances[fname] = float(np.mean(drops)) if drops else 0.0
    return importances


PERM_IMP_RESULTS = {}
for r in tqdm(RESULTS, desc='Phase H — attribution', unit='run'):
    run_id     = r['run_id']
    feat_cols  = r['feat_cols']
    target_col = r['target']
    cache_perm = ARTS_DIR / f'perm_imp_{run_id}.json'

    if cache_perm.exists():
        with open(cache_perm) as f:
            PERM_IMP_RESULTS[run_id] = json.load(f)
        print(f'[{run_id}] loaded from cache.')
        continue

    model_path = ARTS_DIR / f'model_{run_id}.pt'
    if not model_path.exists():
        print(f'[{run_id}] model not found — skipping.'); continue

    _, vl_loader, _, n_feat = make_loaders(feat_cols, target_col, stride=1)
    model = MambaClassifier(n_features=n_feat).to(DEVICE)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))

    imp = compute_sequence_permutation_importance(
        model, vl_loader, base_auc=r['best_auc'],
        device=DEVICE, feature_names=feat_cols, n_repeats=3,
    )
    PERM_IMP_RESULTS[run_id] = imp
    with open(cache_perm, 'w') as f:
        json.dump(imp, f, indent=2)
    print(f'[{run_id}] Top 5: {sorted(imp, key=imp.get, reverse=True)[:5]}')

---
## Phase I — Visualizations

In [ ]:
n_runs = len(RESULTS)
if n_runs > 0:
    cols_ = min(4, n_runs); rows_ = math.ceil(n_runs / cols_)
    fig, axes = plt.subplots(rows_, cols_, figsize=(cols_ * 4, rows_ * 3.5), squeeze=False)
    for ax, r in zip(axes.flatten(), RESULTS):
        ep = range(1, len(r['train_aucs']) + 1)
        ax.plot(ep, r['train_aucs'], color='#2196F3', lw=1.2, label='Train')
        ax.plot(ep, r['val_aucs'],   color='#FF6F00', lw=1.2, label='Val')
        ax.axhline(r['best_auc'], color='green', lw=0.8, ls='--',
                   label=f'Best={r["best_auc"]:.4f}@{r["best_epoch"]}')
        ax.set_title(r['run_id'].replace('__', '\n').replace('label_', ''), fontsize=8)
        ax.legend(fontsize=7); ax.set_ylim(0.45, 0.75); ax.grid(alpha=0.3)
    for ax in axes.flatten()[n_runs:]: ax.set_visible(False)
    fig.suptitle('Mamba v3 — Training Curves', fontsize=12, fontweight='bold')
    fig.tight_layout()
    fig.savefig(ARTS_DIR / 'training_curves.png', dpi=130, bbox_inches='tight')
    plt.show()

In [ ]:
if PERM_IMP_RESULTS and best_run:
    all_feats  = sorted(set(f for imp in PERM_IMP_RESULTS.values() for f in imp))
    imp_matrix = pd.DataFrame(
        {rid: [imp.get(f, 0) for f in all_feats] for rid, imp in PERM_IMP_RESULTS.items()},
        index=all_feats,
    )
    fig, axes = plt.subplots(1, 2, figsize=(14, max(6, len(all_feats) * 0.3)))
    sns.heatmap(imp_matrix, ax=axes[0], cmap='RdYlGn', center=0, annot=True, fmt='.4f',
                linewidths=0.5, cbar_kws={'label': 'AUC Drop'})
    axes[0].set_title('Permutation Importance — AUC Drop per Feature')
    axes[0].tick_params(labelsize=7)
    best_imp  = PERM_IMP_RESULTS.get(best_run['run_id'], {})
    sorted_f  = sorted(best_imp, key=best_imp.get, reverse=True)
    axes[1].barh(range(len(sorted_f)), [best_imp[f] for f in sorted_f],
                 color=['#2E7D32' if best_imp[f] > 0 else '#C62828' for f in sorted_f])
    axes[1].set_yticks(range(len(sorted_f)))
    axes[1].set_yticklabels(sorted_f, fontsize=8)
    axes[1].invert_yaxis(); axes[1].set_xlabel('AUC Drop')
    axes[1].set_title(f'Best: {best_run["run_id"]}')
    fig.tight_layout()
    fig.savefig(ARTS_DIR / 'perm_importance.png', dpi=130, bbox_inches='tight')
    plt.show()

---
## Phase J — Save Results

In [ ]:
out = {
    'notebook': '03_mamba_omni_0fee_v3', 'version': 'v3',
    'interval': INTERVAL, 'created': pd.Timestamp.now().isoformat(),
    'device': str(DEVICE), 'seq_len': SEQ_LEN, 'stride': STRIDE, 'epochs': EPOCHS,
    'scan_chunk': SCAN_CHUNK,
    'feature_pool': {'v1': len(V1_FEATURE_COLS), 'v4': len(V4_FEATURE_COLS),
                     'total': len(ALL_FEATURE_COLS)},
    'selected_features': SELECTED_FEATURES,
    'results': [{k: v for k, v in r.items() if k not in ('val_probs','train_aucs','val_aucs')}
                for r in RESULTS],
    'permutation_importance': PERM_IMP_RESULTS,
}
out_path = ARTS_DIR / 'feature_importance.json'
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2, default=str)
print(f'Saved: {out_path}')
print('\n── Final Summary ─────────────────────────────────────────')
for r in sorted(RESULTS, key=lambda x: x['best_auc'], reverse=True):
    print(f'  {r["run_id"]:<55s}  AUC={r["best_auc"]:.4f}  ep={r["best_epoch"]}  '
          f'time={r["elapsed_s"]/60:.1f}min')